# Part 2: Named Entity Recognition using Hidden Markov Model

This notebook implements a Hidden Markov Model (HMM) for Named Entity Recognition using the Word2Vec embeddings learned in Part 1.

In [ ]:
import numpy as np
from datasets import load_dataset
import json
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

## 1. Load Word Embeddings from Part 1

In [ ]:
# Load embeddings
embeddings = np.load('word_embeddings.npy')

with open('word2idx.json', 'r') as f:
    word2idx_raw = json.load(f)

word2idx = {word: int(idx) for word, idx in word2idx_raw.items()}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f"Loaded embeddings: {embeddings.shape}")
print(f"Vocabulary size: {len(word2idx)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

embedding_dim = embeddings.shape[1]

## 2. Load and Prepare NER Dataset

In [ ]:
dataset = load_dataset('lhoestq/conll2003')

print("Dataset loaded:")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

# Get tag names
try:
    ner_feature = dataset['train'].features['ner_tags']
    if hasattr(ner_feature, 'feature') and hasattr(ner_feature.feature, 'names'):
        tag_names = ner_feature.feature.names
    elif hasattr(ner_feature, 'names'):
        tag_names = ner_feature.names
    else:
        raise AttributeError("Cannot find names attribute")
except (AttributeError, KeyError):
    tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

print("\nNER Tag Names:")
for idx, tag in enumerate(tag_names):
    print(f"  {idx}: {tag}")

num_tags = len(tag_names)
tag2idx = {tag: idx for idx, tag in enumerate(tag_names)}
idx2tag = {idx: tag for idx, tag in enumerate(tag_names)}

## 3. Prepare Data for HMM

In [ ]:
def get_word_embedding(word):
    """Get embedding for a word, return zero vector if not found"""
    word_lower = word.lower()
    if word_lower in word2idx:
        idx = word2idx[word_lower]
        return embeddings[idx]
    else:
        # Return zero vector for unknown words
        return np.zeros(embedding_dim)

def prepare_hmm_data(split_data):
    """Prepare data for HMM: list of (embeddings, tags) sequences"""
    sequences = []
    
    for sample in split_data:
        tokens = sample['tokens']
        ner_tags = sample['ner_tags']
        
        # Get embeddings for each token
        word_embeddings = [get_word_embedding(token) for token in tokens]
        
        sequences.append({
            'embeddings': np.array(word_embeddings),
            'tags': ner_tags,
            'tokens': tokens
        })
    
    return sequences

# Prepare training and test data
train_sequences = prepare_hmm_data(dataset['train'])
test_sequences = prepare_hmm_data(dataset['test'])

print(f"Prepared {len(train_sequences)} training sequences")
print(f"Prepared {len(test_sequences)} test sequences")
print(f"\nExample sequence:")
print(f"  Tokens: {train_sequences[0]['tokens']}")
print(f"  Tags: {train_sequences[0]['tags']}")
print(f"  Embeddings shape: {train_sequences[0]['embeddings'].shape}")

## 4. HMM Model Implementation

We implement an HMM with:
- **Transition probabilities**: P(tag_i | tag_{i-1})
- **Emission probabilities**: P(embedding | tag) modeled as Gaussian distributions
- **Initial probabilities**: P(tag) for the first token

In [ ]:
class HMM_NER:
    def __init__(self, num_tags, tag_names, embedding_dim):
        self.num_tags = num_tags
        self.tag_names = tag_names
        self.embedding_dim = embedding_dim
        
        # Initialize probability matrices
        self.transition_probs = np.zeros((num_tags, num_tags))  # P(tag_j | tag_i)
        self.initial_probs = np.zeros(num_tags)  # P(tag) for first token
        
        # For emission probabilities, we'll use Gaussian distributions
        # Store mean and covariance for each tag
        self.emission_means = {}  # mean embedding for each tag
        self.emission_covs = {}   # covariance matrix for each tag
        
        # Smoothing parameter for transition probabilities
        self.smoothing = 1e-10
        
    def train(self, sequences):
        """Train HMM from sequences"""
        print("Training HMM...")
        
        # Count transitions and initial tags
        transition_counts = np.zeros((self.num_tags, self.num_tags))
        initial_counts = np.zeros(self.num_tags)
        
        # Collect embeddings for each tag
        tag_embeddings = defaultdict(list)
        
        for seq in sequences:
            tags = seq['tags']
            embs = seq['embeddings']
            
            # Count initial tag
            initial_counts[tags[0]] += 1
            
            # Count transitions and collect embeddings
            for i in range(len(tags)):
                tag = tags[i]
                emb = embs[i]
                
                # Collect embedding for this tag
                tag_embeddings[tag].append(emb)
                
                # Count transition (if not last token)
                if i < len(tags) - 1:
                    next_tag = tags[i + 1]
                    transition_counts[tag, next_tag] += 1
        
        # Compute transition probabilities with smoothing
        for i in range(self.num_tags):
            total = transition_counts[i].sum() + self.num_tags * self.smoothing
            if total > 0:
                self.transition_probs[i] = (transition_counts[i] + self.smoothing) / total
            else:
                self.transition_probs[i] = 1.0 / self.num_tags
        
        # Compute initial probabilities
        total_initial = initial_counts.sum()
        self.initial_probs = initial_counts / total_initial
        
        # Compute emission parameters (Gaussian)
        for tag in range(self.num_tags):
            if tag in tag_embeddings and len(tag_embeddings[tag]) > 0:
                tag_embs = np.array(tag_embeddings[tag])
                
                # Compute mean
                self.emission_means[tag] = np.mean(tag_embs, axis=0)
                
                # Compute covariance with regularization
                if len(tag_embs) > 1:
                    cov = np.cov(tag_embs.T)
                    # Add small value to diagonal for numerical stability
                    cov += np.eye(self.embedding_dim) * 1e-6
                    self.emission_covs[tag] = cov
                else:
                    # Use identity matrix if only one sample
                    self.emission_covs[tag] = np.eye(self.embedding_dim)
            else:
                # No observations for this tag - use zero mean and identity covariance
                self.emission_means[tag] = np.zeros(self.embedding_dim)
                self.emission_covs[tag] = np.eye(self.embedding_dim)
        
        print("Training complete!")
        print(f"\nModel statistics:")
        print(f"  Initial probabilities: {self.initial_probs[:5]}...")
        print(f"  Transition matrix shape: {self.transition_probs.shape}")
        print(f"  Number of emission distributions: {len(self.emission_means)}")
        
    def emission_probability(self, embedding, tag):
        """Compute P(embedding | tag) using Gaussian distribution"""
        try:
            mean = self.emission_means[tag]
            cov = self.emission_covs[tag]
            
            # Use multivariate normal PDF
            # For numerical stability, we work in log space
            prob = multivariate_normal.pdf(embedding, mean=mean, cov=cov, allow_singular=True)
            
            # Avoid log(0)
            return max(prob, 1e-300)
        except:
            # Fallback to very small probability
            return 1e-300
    
    def viterbi(self, embeddings):
        """
        Viterbi algorithm for finding most likely tag sequence.
        
        Args:
            embeddings: numpy array of shape (seq_len, embedding_dim)
        
        Returns:
            best_path: list of tag indices
        """
        seq_len = len(embeddings)
        
        # Initialize DP table (log probabilities for numerical stability)
        # viterbi[t][tag] = max log probability of path ending in 'tag' at time t
        viterbi = np.full((seq_len, self.num_tags), -np.inf)
        backpointer = np.zeros((seq_len, self.num_tags), dtype=int)
        
        # Initialize first time step
        for tag in range(self.num_tags):
            emission_prob = self.emission_probability(embeddings[0], tag)
            initial_prob = self.initial_probs[tag]
            
            viterbi[0][tag] = np.log(initial_prob + 1e-300) + np.log(emission_prob)
        
        # Forward pass
        for t in range(1, seq_len):
            for curr_tag in range(self.num_tags):
                # Compute emission probability
                emission_prob = self.emission_probability(embeddings[t], curr_tag)
                log_emission = np.log(emission_prob)
                
                # Find best previous tag
                best_prob = -np.inf
                best_prev_tag = 0
                
                for prev_tag in range(self.num_tags):
                    transition_prob = self.transition_probs[prev_tag][curr_tag]
                    prob = viterbi[t-1][prev_tag] + np.log(transition_prob + 1e-300) + log_emission
                    
                    if prob > best_prob:
                        best_prob = prob
                        best_prev_tag = prev_tag
                
                viterbi[t][curr_tag] = best_prob
                backpointer[t][curr_tag] = best_prev_tag
        
        # Backward pass - reconstruct best path
        best_path = [0] * seq_len
        best_path[-1] = np.argmax(viterbi[-1])
        
        for t in range(seq_len - 2, -1, -1):
            best_path[t] = backpointer[t + 1][best_path[t + 1]]
        
        return best_path
    
    def predict(self, sequences):
        """Predict tags for a list of sequences"""
        predictions = []
        
        for seq in sequences:
            embeddings = seq['embeddings']
            predicted_tags = self.viterbi(embeddings)
            predictions.append(predicted_tags)
        
        return predictions

print("HMM_NER class defined successfully")

## 5. Train the HMM Model

In [ ]:
# Create and train HMM
hmm = HMM_NER(num_tags=num_tags, tag_names=tag_names, embedding_dim=embedding_dim)
hmm.train(train_sequences)

## 6. Evaluate on Test Set

In [ ]:
print("Making predictions on test set...")
predictions = hmm.predict(test_sequences)

# Flatten predictions and true labels for evaluation
all_true_labels = []
all_predictions = []

for i, seq in enumerate(test_sequences):
    true_tags = seq['tags']
    pred_tags = predictions[i]
    
    all_true_labels.extend(true_tags)
    all_predictions.extend(pred_tags)

print(f"Total tokens evaluated: {len(all_true_labels)}")

## 7. Compute Metrics

In [ ]:
# Compute accuracy
accuracy = accuracy_score(all_true_labels, all_predictions)

# Compute precision, recall, F1
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    all_true_labels, all_predictions, average='weighted', zero_division=0
)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    all_true_labels, all_predictions, average='macro', zero_division=0
)

print("\n" + "="*70)
print("HMM-based NER Results on Test Set")
print("="*70)
print(f"\nWeighted Average Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision_weighted:.4f}")
print(f"  Recall:    {recall_weighted:.4f}")
print(f"  F1-Score:  {f1_weighted:.4f}")

print(f"\nMacro Average Metrics:")
print(f"  Precision: {precision_macro:.4f}")
print(f"  Recall:    {recall_macro:.4f}")
print(f"  F1-Score:  {f1_macro:.4f}")

## 8. Detailed Classification Report

In [ ]:
print("\nDetailed Classification Report:")
print(classification_report(
    all_true_labels,
    all_predictions,
    target_names=tag_names,
    zero_division=0
))

## 9. Per-Class Metrics

In [ ]:
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
    all_true_labels, all_predictions, labels=range(num_tags), zero_division=0
)

print("Per-Class Metrics:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 70)
for i in range(num_tags):
    print(f"{tag_names[i]:<15} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support_per_class[i]:<10}")

## 10. Example Predictions

In [ ]:
# Show example predictions
num_examples = 3

for idx in range(num_examples):
    seq = test_sequences[idx]
    tokens = seq['tokens']
    true_tags = seq['tags']
    pred_tags = predictions[idx]
    
    print(f"\nExample {idx + 1}:")
    print(f"Tokens: {tokens}")
    print(f"\nTrue tags:")
    print(f"  {[tag_names[t] for t in true_tags]}")
    print(f"\nPredicted tags:")
    print(f"  {[tag_names[p] for p in pred_tags]}")
    
    correct = sum(p == t for p, t in zip(pred_tags, true_tags))
    print(f"\nCorrect predictions: {correct}/{len(tokens)} ({100*correct/len(tokens):.1f}%)")
    print("-" * 70)

## 11. Comparison with Feed-Forward Neural Network

Let's compare the HMM results with the Feed-Forward Neural Network from the previous notebook.

In [ ]:
# Results from Feed-Forward NN (from ner_neural_network.ipynb)
ffnn_results = {
    'accuracy': 0.8990,
    'precision_weighted': 0.8825,
    'recall_weighted': 0.8990,
    'f1_weighted': 0.8822,
    'precision_macro': 0.7212,
    'recall_macro': 0.5036,
    'f1_macro': 0.5772
}

# Current HMM results
hmm_results = {
    'accuracy': accuracy,
    'precision_weighted': precision_weighted,
    'recall_weighted': recall_weighted,
    'f1_weighted': f1_weighted,
    'precision_macro': precision_macro,
    'recall_macro': recall_macro,
    'f1_macro': f1_macro
}

print("\n" + "="*70)
print("Comparison: Feed-Forward NN vs HMM")
print("="*70)
print(f"\n{'Metric':<25} {'Feed-Forward NN':<20} {'HMM':<20} {'Difference':<15}")
print("-" * 80)

for metric in ['accuracy', 'f1_weighted', 'f1_macro']:
    ffnn_val = ffnn_results[metric]
    hmm_val = hmm_results[metric]
    diff = hmm_val - ffnn_val
    diff_str = f"{diff:+.4f}"
    
    print(f"{metric:<25} {ffnn_val:<20.4f} {hmm_val:<20.4f} {diff_str:<15}")

print("\n" + "="*70)
print("\nKey Observations:")
print("  • Feed-Forward NN: Processes each token independently using learned features")
print("  • HMM: Uses probabilistic sequence model with transition probabilities")
print("  • Both models use the same Word2Vec embeddings from Part 1")
print("="*70)

## Summary

This notebook implemented a Hidden Markov Model for NER with:
1. **Transition probabilities** learned from training data
2. **Emission probabilities** modeled as Gaussian distributions over embeddings
3. **Viterbi algorithm** for finding the most likely tag sequence
4. Complete evaluation with accuracy, precision, recall, and F1-score
5. Comparison with the Feed-Forward Neural Network approach